💡 **Environment:** `clamp-analyses`

# ARCHS4 CRISPR-Cas9 - Gene Enrichment Analysis

This notebook scores every CLAMP latent variable (LV) trained on ARCHS4 against two gene sets derived from a CRISPR-Cas9 lipid perturbation screen: genes whose knockout strongly increases lipid accumulation and genes whose knockout decreases it. Because `fgsea` produces slightly different p-values across runs, each LV x gene-set pair is evaluated 10 times and the conservative (maximum) p-value is used.

## Libraries

In [ ]:
library(here)
library(dplyr)
library(purrr)
library(tidyr)
library(tibble)
library(stringr)
library(readr)
library(fgsea)
library(ggplot2)
library(msigdbr)

## Input

In [ ]:
archs4_CLAMPfull <- readRDS(here('output/01_model_building/04_archs4/06_bp_coverage_rshall/06_bp_coverage_hall_rs_100/CLAMPfull_hall.rds'))

In [ ]:
archs4_CLAMPfull_Z <- data.frame(as.matrix(archs4_CLAMPfull$Z))

In [ ]:
head(archs4_CLAMPfull_Z)

In [ ]:
archs4_CLAMPfull_summary <- data.frame(as.matrix(archs4_CLAMPfull$summary))

In [ ]:
archs4_CLAMPfull_summary <- archs4_CLAMPfull_summary %>% 
dplyr::mutate(FDR = as.numeric(FDR)) %>%
dplyr::mutate(AUC = as.numeric(AUC))

In [ ]:
head(archs4_CLAMPfull_summary)

In [ ]:
archs4_CLAMPfull_summary_sig <- archs4_CLAMPfull_summary %>% 
dplyr::filter(FDR < 0.05) %>%
dplyr::filter(AUC > 0.7)

archs4_CLAMPfull_summary_sig %>% 
pull(LV) %>% 
unique() %>% 
length()

In [ ]:
archs4_CLAMPfull <- NULL

## Output

In [ ]:
output_dir <- here("output/03_model_biology/00_archs4/01_CRISPRCas9/00_gene_enrichment_CRISPRCas9")
dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)

## Lipids gene sets

In [ ]:
lipid_deg <- read.csv(
  here("data/archs4/crispr_cas9/lipid_DEG.csv"),
  stringsAsFactors = FALSE
)

head(lipid_deg)

In [ ]:
orig_deg_gene_sets <- list()
for (r in unique(lipid_deg$rank)) {
  if (r == 0) next
  genes <- lipid_deg$gene_name[lipid_deg$rank == r]
  orig_deg_gene_sets[[paste0("gene_set_", r)]] <- genes
}

In [ ]:
length(orig_deg_gene_sets)

## Combine gene sets into "increase lipids" and "decrease lipids"

In [ ]:
deg_gene_sets <- list()

In [ ]:
# genes that increase lipids (rank == 3: strongest effect, both conditions)
deg_gene_sets[["gene_set_increase"]] <- orig_deg_gene_sets[["gene_set_3"]]

In [ ]:
# genes that decrease lipids (rank == -3: strongest effect, both conditions)
deg_gene_sets[["gene_set_decrease"]] <- orig_deg_gene_sets[["gene_set_-3"]]

In [ ]:
length(deg_gene_sets)

In [ ]:
stopifnot(length(deg_gene_sets[["gene_set_increase"]]) == 6)

In [ ]:
length(deg_gene_sets[["gene_set_decrease"]])

In [ ]:
stopifnot(length(deg_gene_sets[["gene_set_decrease"]]) == 8)

In [ ]:
# test new increase set
new_set <- deg_gene_sets[["gene_set_increase"]]
expected_set <- orig_deg_gene_sets[["gene_set_3"]]

stopifnot(length(new_set) == length(unique(new_set)))

stopifnot(
  length(new_set) ==
    length(
      intersect(
        new_set,
        expected_set
      )
    )
)

In [ ]:
# test new decrease set
new_set <- deg_gene_sets[["gene_set_decrease"]]
expected_set <- orig_deg_gene_sets[["gene_set_-3"]]

stopifnot(length(new_set) == length(unique(new_set)))

stopifnot(
  length(new_set) ==
    length(
      intersect(
        new_set,
        expected_set
      )
    )
)
     

## Prepare LVs list

In [ ]:
lvs <- list()
z_gene_names <- rownames(archs4_CLAMPfull_Z)

for (cidx in 1:ncol(archs4_CLAMPfull_Z)) {
  data <- archs4_CLAMPfull_Z[, cidx]
  names(data) <- z_gene_names

  lvs[[paste0("LV", cidx)]] <- data
}

In [ ]:
length(lvs)

## Compute enrichment on all LVs

Since fgsea generates slightly different p-values across the same runs, here I run it 10 times for each LV/gene-set pair, and later I will take the maximum p-value. This is to avoid reproducibility/inconsistency problems.

In [ ]:
n_reps <- 10
set.seed(0)

In [ ]:
csv_path <- file.path(output_dir, "fgsea_hi_conf_all_lvs.csv")

if (file.exists(csv_path)) {
  message("fgsea results already exist: skipping loop.")
  results <- NULL
} else {
  results <- list()

  for (lv in names(lvs)) {
    repetitions <- list()

    for (i in 1:n_reps) {
      rep_res <- fgsea(pathways = deg_gene_sets, stats = lvs[[lv]], scoreType = "pos", eps = 0.0)[order(pval), ]
      rep_res[, "lv"] <- lv
      rep_res[, "rep_idx"] <- i

      repetitions[[i]] <- rep_res
    }

    res <- do.call(rbind, repetitions)

    results[[lv]] <- res
  }
}

In [ ]:
if (is.null(results)) {
  df <- read.csv(csv_path, stringsAsFactors = FALSE)
} else {
  df <- do.call(rbind, results)
}

In [ ]:
if (!is.null(results)) {
  df <- df %>% mutate(leadingEdge = map_chr(leadingEdge, toString))
}

In [ ]:
dim(df)

In [ ]:
head(df)

In [ ]:
df_sig <- df %>% 
dplyr::filter(padj < 0.05)

dim(df_sig)
head(df_sig)

length(unique(df_sig$lv))

## Save

In [ ]:
if (!is.null(results)) {
  write.csv(df, csv_path, row.names = FALSE)
}